# Step 2 — Tokenizer, Padding & Class Weights
Neural networks can't read words, only numbers. This notebook turns every complaint into a fixed-length list of integers, and prepares class weights for the imbalance.

**Pipeline:** text → tokens → integer ids → padded matrix `(n_samples, MAX_LEN)` → (next step) Embedding layer.

## 2.1 Load the splits from Step 1

In [10]:
import os, glob, json
import numpy as np
import pandas as pd
from collections import Counter
from keras.utils import pad_sequences
from sklearn.utils.class_weight import compute_class_weight

# Find the splits folder (same working dir as Step 1, or a Kaggle dataset you saved it into)
cands = glob.glob('splits/train.csv') + glob.glob('/kaggle/working/splits/train.csv') + glob.glob('/kaggle/input/**/splits/train.csv', recursive=True)
SPLIT_DIR = os.path.dirname(cands[0])
print('Using splits from:', SPLIT_DIR)

train_df = pd.read_csv(f'{SPLIT_DIR}/train.csv')
val_df   = pd.read_csv(f'{SPLIT_DIR}/val.csv')
test_df  = pd.read_csv(f'{SPLIT_DIR}/test.csv')
label2id = json.load(open(f'{SPLIT_DIR}/label2id.json'))
print(len(train_df), len(val_df), len(test_df))

Using splits from: /kaggle/input/notebooks/mazenayad/step-1-data-cleaning-split/splits
86914 18625 18625


## 2.2 Build the vocabulary — from TRAIN only
If we count words on val/test too, information from the test set leaks into the model. So the vocabulary is built **only from train**; any unseen word in val/test becomes `<OOV>`.

Special ids: `0 = <PAD>` (padding), `1 = <OOV>` (unknown word).

First, how big should the vocabulary be? Let's see how much of the text each size covers.

In [11]:
word_counts = Counter(w for text in train_df['narrative'] for w in text.split())
total_tokens = sum(word_counts.values())
print('Unique words in train:', len(word_counts))

for k in [5_000, 10_000, 20_000, 30_000, 50_000]:
    covered = sum(c for _, c in word_counts.most_common(k)) / total_tokens
    print(f'top {k:>6,} words cover {covered:.2%} of all tokens')

Unique words in train: 39153
top  5,000 words cover 97.38% of all tokens
top 10,000 words cover 99.09% of all tokens
top 20,000 words cover 99.71% of all tokens
top 30,000 words cover 99.88% of all tokens
top 50,000 words cover 100.00% of all tokens


Pick `VOCAB_SIZE` where coverage is high but the vocab stays small (rare words just add noise and parameters). We'll use **20,000** unless the table above shows a clearly better knee.

In [12]:
VOCAB_SIZE = 20_000          # includes PAD and OOV
PAD_ID, OOV_ID = 0, 1

most_common = [w for w, _ in word_counts.most_common(VOCAB_SIZE - 2)]
word_index = {w: i + 2 for i, w in enumerate(most_common)}   # ids start at 2
print('Vocabulary size:', len(word_index) + 2)
print('Example:', list(word_index.items())[:8])

def texts_to_ids(texts):
    return [[word_index.get(w, OOV_ID) for w in t.split()] for t in texts]

train_seq = texts_to_ids(train_df['narrative'])
val_seq   = texts_to_ids(val_df['narrative'])
test_seq  = texts_to_ids(test_df['narrative'])

print('\nOriginal :', train_df['narrative'].iloc[0][:100])
print('As ids   :', train_seq[0][:15])

Vocabulary size: 20000
Example: [('account', 2), ('credit', 3), ('report', 4), ('payment', 5), ('information', 6), ('time', 7), ('would', 8), ('loan', 9)]

Original : loan depot charge appraisal fee wit consent told refinance
As ids   : [9, 1657, 45, 691, 69, 3519, 572, 18, 417]


## 2.3 Padding — and why *pre*-padding
Every sequence must have the same length to be batched. We choose `MAX_LEN` from the length distribution (95th percentile ≈ 253 words) → **250**.

- Shorter texts get zeros added; longer texts get cut.
- **`padding='pre'`** (zeros at the START): an RNN reads left→right and its hidden state at the *last* step is what gets classified. With post-padding, the RNN processes a long run of zeros after the real words and its memory of the text fades — SimpleRNN often fails to learn (this is the issue one of the posts you sent mentioned: ~27% accuracy).
- **`truncating='post'`**: for long texts keep the beginning, where the complaint usually states the problem.

In [13]:
MAX_LEN = 250

def pad(seqs):
    return pad_sequences(seqs, maxlen=MAX_LEN, padding='pre', truncating='post', value=PAD_ID).astype('int32')

X_train, X_val, X_test = pad(train_seq), pad(val_seq), pad(test_seq)
y_train, y_val, y_test = train_df['label'].values, val_df['label'].values, test_df['label'].values

print('X_train:', X_train.shape, '| X_val:', X_val.shape, '| X_test:', X_test.shape)

lens = np.array([len(s) for s in train_seq])
print(f'Truncated texts (> {MAX_LEN} words): {(lens > MAX_LEN).mean():.1%}')
print(f'OOV rate in test: {np.mean(np.array(sum(test_seq, [])) == OOV_ID):.2%}')
print('\nExample row (first 10 are padding zeros if the text was short):')
print(X_train[0][:12], '...', X_train[0][-8:])

X_train: (86914, 250) | X_val: (18625, 250) | X_test: (18625, 250)
Truncated texts (> 250 words): 5.1%
OOV rate in test: 0.43%

Example row (first 10 are padding zeros if the text was short):
[0 0 0 0 0 0 0 0 0 0 0 0] ... [1657   45  691   69 3519  572   18  417]


## 2.4 Class weights (handling the imbalance)
`credit_reporting` is 45% of the data, `retail_banking` ~11%. Without correction, the model is rewarded for favouring the big class. **Class weights** make a mistake on a small class cost more during training.

In [14]:
classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight = {int(c): float(w) for c, w in zip(classes, weights)}

id2label = {v: k for k, v in label2id.items()}
for c, w in class_weight.items():
    print(f'{id2label[c]:22s} weight = {w:.2f}')

credit_card            weight = 1.66
credit_reporting       weight = 0.44
debt_collection        weight = 1.18
mortgages_and_loans    weight = 1.33
retail_banking         weight = 1.85


## 2.5 Save everything for the model notebooks

In [15]:
os.makedirs('processed', exist_ok=True)
np.savez_compressed('processed/data.npz',
                    X_train=X_train, y_train=y_train,
                    X_val=X_val,     y_val=y_val,
                    X_test=X_test,   y_test=y_test)

json.dump({'word_index': word_index, 'vocab_size': VOCAB_SIZE, 'max_len': MAX_LEN,
           'pad_id': PAD_ID, 'oov_id': OOV_ID, 'padding': 'pre', 'truncating': 'post'},
          open('processed/tokenizer.json', 'w'))
json.dump({'class_weight': class_weight, 'label2id': label2id}, open('processed/meta.json', 'w'))

print(os.listdir('processed'))

['meta.json', 'tokenizer.json', 'data.npz']
